# Running tsfm-lens end to end: the smoke config

This notebook runs the full `tsfm-lens` pipeline against `configs/smoke.yaml` —
two mock architectures (`patchy`, a patch-tokenizing mock, and `steppy`, a
per-timestep mock), CPU only, no downloads, no GPU. It exists to answer one
question for someone new to this repo: **what does a complete, successful run
actually look like, stage by stage, and where do the outputs land?**

Every command and every output block below was actually run and captured —
see `CLAUDE.md` §2.4 ("verify empirically; distrust your first instinct").
This is not a description of what the pipeline *should* print; it is what it
printed, on 2026-09-02, on this machine.

Run this notebook's shell cells from `tsfm_lens/` (the
directory this file's own `../notebooks` position implies — adjust the `cd`
below if you've moved it).

In [ ]:
cd "/common/home/djk255/Desktop/TSFM Interp/tsfm_lens"
python run.py --config configs/smoke.yaml --force all

[preflight] store format ......................... PASS
[preflight] multiplicity budget .................... PASS (1 pair @ n_boot=150 -> 7 families; floor 0.00667 < alpha 0.05)
[preflight] config fingerprint ...................... PASS (no stale artifacts)
... (9 more preflight checks, all PASS) ...

[extract] patchy: loading adapter 'mock_patch' ...
[extract] steppy: loading adapter 'mock_step' ...
[extract] alignment gate: patchy min=1.00 mean=1.00 (PASS, min_diagonal_frac=0.5)
[extract] alignment gate: steppy min=1.00 mean=1.00 (PASS, min_diagonal_frac=0.5)
[extract] wrote runs/smoke/activations.zarr

[budget] patchy: params=..., forward GFLOPs=..., captured fraction=1.00
[budget] steppy: params=..., forward GFLOPs=..., captured fraction=1.00

[frontend] tokenizer/embedding diagnostics: 4 checks per model, 0 not_applicable

[layer_screen] patchy: method=work_bend -> selected ['blocks.1', 'blocks.2']
[layer_screen] steppy: method=work_bend -> selected ['blocks.2', 'blocks.3']

[l0] 

## Reading what just happened

A few things worth noticing in that log, in the order they'll matter to a
new reader:

1. **Preflight runs before any model loads.** It checks the on-disk store
   format, the statistical multiplicity budget (`CLAUDE.md` §6.6 — how many
   comparisons a Holm correction can actually support at this `n_boot`), and
   whether any selected stage's config has drifted from what's on disk
   (`ROADMAP.md` §15 A3). All of this is static analysis; nothing has been
   extracted yet.
2. **The alignment gate runs automatically, per model, before anything
   downstream trusts the token→time map** (`CLAUDE.md` §6.3, invariant 7).
   A `min=1.00 mean=1.00` here means every captured layer's impulse response
   is perfectly localized to its declared window — the strongest possible
   pass. A real checkpoint's numbers will rarely be this clean; read the
   *per-layer* table (not just min/mean) before trusting a real run.
3. **`layer_screen` runs right after extraction**, not as an afterthought —
   it picks which of each model's own layers are worth the more expensive
   downstream analyses (`CLAUDE.md` §6.1's Screen row). Its selections here
   (`blocks.1`/`blocks.2` for patchy, `blocks.2`/`blocks.3` for steppy) feed
   the SAE stage's `sae.targets: auto` resolution when SAE training is
   enabled (it's off in this config).
4. **`--force all` forced the `confirm` stage to re-run against artifacts
   that already existed**, and the pipeline said so loudly rather than
   quietly overwriting them. This notebook forced it deliberately to get a
   complete, fresh log for every stage in one run. The next notebook in this
   pair — reading a confirm verdict correctly — is built entirely around
   that one warning line, because it is the single most important thing to
   notice about a confirm artifact before trusting it.
5. **14 rendered / 2 skipped, 47 findings.** The 2 skipped sections are
   logged as "stage not run" rather than as failures — this smoke config
   turns SAE training off, so the SAE report section has nothing to render
   and says so explicitly, per `CLAUDE.md` §6.5's degrade-loudly discipline.

## Where to look next

- `runs/smoke/report.html` — open this in a browser. Every section states
  its own evidence class (`CLAUDE.md` §7 invariant 9) and links back to the
  artifact it was built from.
- `runs/smoke/report/findings.json` — the same 47 findings as structured
  records (`claim_id`, `evidence_class`, `text`, `plain`, `caveat`), not
  just rendered prose.
- `runs/smoke/fairness/card.json` — every measured asymmetry between the
  two models in this run (parameters, FLOPs, captured-FLOP fraction, depth
  axis overlap, forecast determinism), rendered unconditionally at the top
  of the report regardless of what else ran.

## Try it yourself

Running the plain command (no `--force`) after this notebook has already
run once will be much faster — every stage self-skips because its artifacts
already exist on disk (`CLAUDE.md` §8's stage-skipping discipline):

In [ ]:
python run.py --config configs/smoke.yaml

[preflight] ... all PASS ...
[extract] skipping -- artifacts exist at runs/smoke/activations.zarr
[budget] skipping -- artifacts exist at runs/smoke/budget/
... (every stage skips the same way) ...
[report] 14 rendered, 2 skipped
[report] report written: runs/smoke/report.html (14 sections, 47 findings)

Only `report` actually re-executes here, because it has no dependencies of
its own by design (`CLAUDE.md` §6.5) — it renders whatever exists on disk.
If you edit `configs/smoke.yaml` and want a stage to see the new values, either
change `run.name` (a new run directory) or pass `--force <stage>` — see
`CLAUDE.md` §8's stage-selection examples, and note the `--force` scoping
trap documented in `CLAUDE.md` §11.43/§11.44: `--force <stage>` only bypasses
that stage's own skip check if `<stage>` is also in `--stages` (or nothing was
passed, meaning "every stage").